# Perf Analysis (Part C)

Reads a `metrics.csv` produced by `perf.load_test` and plots latency, TTFT, and TPOT distributions. Commentary at the bottom is ≤200 words.

Regenerate data with:

```bash
python -m perf.load_test --num-requests 200 --concurrency 16 --mode stream \
    --output results/perf/metrics.csv
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from perf.metrics import load_metrics, summarize

sns.set_theme(context="notebook", style="whitegrid")

METRICS_CSV = Path("../results/perf/metrics.csv")
if not METRICS_CSV.exists():
    # Fallback to repo-root metrics.csv (default path of perf.load_test).
    METRICS_CSV = Path("../metrics.csv")
print(f"reading {METRICS_CSV}")
df = load_metrics(METRICS_CSV)
df.head()

In [ ]:
summary = summarize(df.to_dict(orient="records"))
summary.round(4)

## Latency distribution by prompt kind

In [ ]:
ok = df[df["status"] == "ok"]
fig, ax = plt.subplots(figsize=(8, 4))
for kind in ok["prompt_kind"].unique():
    vals = ok[ok["prompt_kind"] == kind]["wall_s"]
    sns.kdeplot(vals, label=kind, fill=True, alpha=0.25, ax=ax)
ax.set_xlabel("end-to-end latency (s)")
ax.set_title("Latency distribution by prompt kind")
ax.legend()
plt.tight_layout()
plt.show()

## Time-to-first-token (stream only)

In [ ]:
stream = ok[(ok["mode"] == "stream") & ok["ttft_s"].notna()]
if stream.empty:
    print("No streaming requests in this run.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=stream, x="prompt_kind", y="ttft_s", ax=ax)
    ax.set_ylabel("TTFT (s)")
    ax.set_title("Time-to-first-token by prompt kind")
    plt.tight_layout()
    plt.show()

## Tokens-per-output-second (TPOT)

In [ ]:
from perf.metrics import compute_tpot

if not ok.empty:
    ok = ok.copy()
    ok["tpot"] = ok.apply(
        lambda r: compute_tpot(float(r["wall_s"]), r["ttft_s"], int(r["output_tokens"] or 0)),
        axis=1,
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=ok, x="prompt_kind", y="tpot", ax=ax)
    ax.set_ylabel("tokens / second (decode)")
    ax.set_title("TPOT by prompt kind")
    plt.tight_layout()
    plt.show()

## Commentary (≤200 words)

The table above summarises a synthetic-load sweep mixing short and long prompts at a configurable concurrency. Key observations to expect on a real vLLM deployment:

1. **Long-prompt TTFT dominates latency.** Prefill is O(input_tokens²/chunk); long prompts spend most of their wall clock before the first output token arrives. Raising `--gpu-memory-utilization` and enabling chunked prefill both cut TTFT materially without hurting throughput.
2. **TPOT is largely prompt-length-independent** once prefill is complete — decode scales with batch, not with per-request prompt size. If TPOT varies with prompt kind, that's usually a memory-pressure signal: KV cache is full, batch size fell, per-request decode lag increased.
3. **P99 worsens super-linearly with concurrency.** vLLM's continuous batching keeps P50 flat but the tail grows when the scheduler starts preempting mid-generation requests. The mitigation is to tune `--max-num-seqs` downward and accept slightly lower throughput for a tighter tail.

All mock-backend runs complete in sub-second wall time because there is no network or decode work; the shape of the distributions is meaningful only against a live vLLM server.